# 02 · Configuración del Modelo de Gobierno — YelpConnect

**Tipo:** Setup / Gobierno de Datos  
**Ubicación:** `00_setup/02_configurar_modelo_gobierno`  
**Catálogo:** `workspace`  
**Schema:** `workspace.yelp_gov`  
**Ejecución:** Idempotente

---

## Propósito

Preparar las estructuras físicas de gobierno utilizadas por los procesos
de **Data Quality** y **Trust & Safety**.

Este notebook crea y documenta:

### Data Quality

- `workspace.yelp_gov.dq_reglas`
- `workspace.yelp_gov.dq_resultados`

### Trust & Safety

- `workspace.yelp_gov.evaluation_rule`
- `workspace.yelp_gov.evaluation_rule_version`
- `workspace.yelp_gov.evaluation_run`

---

## Posición dentro del pipeline

```text
00_reset_rebuild_lakehouse
        ↓
01_configurar_lakehouse
        ↓
02_configurar_modelo_gobierno
        ↓
03_validar_fuentes_raw
        ↓
Catálogos DQ / T&S
        ↓
Bronze / DQ / Silver / T&S / Gold
```

---

## Separación de responsabilidades

| Responsabilidad | Notebook |
|---|---|
| Crear schemas y Volume RAW | `01_configurar_lakehouse` |
| Crear estructuras físicas de gobierno | `02_configurar_modelo_gobierno` |
| Publicar reglas DQ | `01_publicar_catalogo_reglas_dq_silver` |
| Publicar reglas T&S | `02_publicar_catalogo_reglas_ts` |

---

## Este notebook SÍ

- valida `workspace.yelp_gov`;
- crea las tablas operativas de gobierno;
- documenta tablas y columnas;
- aplica metadata y tags cuando están disponibles;
- valida el modelo físico.

## Este notebook NO

- publica reglas `DQ-SL-*`;
- publica reglas `R001-R007`;
- ejecuta reglas DQ;
- ejecuta reglas T&S;
- calcula Risk Score;
- asigna P1/P2/P3;
- modifica Owners;
- ejecuta `GRANT` o `REVOKE`.

In [0]:
# COMMAND ----------

CATALOG = "workspace"
GOV_SCHEMA = "yelp_gov"

GOV = f"{CATALOG}.{GOV_SCHEMA}"

DQ_RULE_TABLE = f"{GOV}.dq_reglas"
DQ_RESULT_TABLE = f"{GOV}.dq_resultados"

RULE_TABLE = f"{GOV}.evaluation_rule"
RULE_VERSION_TABLE = f"{GOV}.evaluation_rule_version"
EVALUATION_RUN_TABLE = f"{GOV}.evaluation_run"

print("=" * 72)
print("YELPCONNECT - MODELO FÍSICO DE GOBIERNO")
print("=" * 72)

print(f"Schema              : {GOV}")
print(f"DQ Rules            : {DQ_RULE_TABLE}")
print(f"DQ Results          : {DQ_RESULT_TABLE}")
print(f"T&S Rules           : {RULE_TABLE}")
print(f"T&S Rule Versions   : {RULE_VERSION_TABLE}")
print(f"T&S Evaluation Runs : {EVALUATION_RUN_TABLE}")

print("=" * 72)

# COMMAND ----------

 ## 1 · Validación del schema de gobierno

El schema:

 `workspace.yelp_gov`

 debe haber sido creado previamente por:

 `01_configurar_lakehouse`
 
 Si no existe, el pipeline está siendo ejecutado fuera de secuencia.

In [0]:
# COMMAND ----------


try:

    spark.sql(
        f"DESCRIBE SCHEMA {GOV}"
    )

    print(
        f"✔ Schema disponible: {GOV}"
    )

except Exception as exc:

    raise Exception(
        f"""
        ❌ No existe el schema:

        {GOV}

        Ejecutar primero:
        00_setup/01_configurar_lakehouse
        """
    ) from exc

# COMMAND ----------

 # Data Quality

 ## 2 · Catálogo de reglas — `dq_reglas`

 Esta tabla almacena la definición gobernada de las reglas Data Quality.

 ### Grano

 **1 fila = 1 regla Data Quality**

 Cada regla define:

 - capa;
 - tabla objetivo;
 - columna;
 - dimensión;
 - tipo de regla;
 - descripción;
 - expresión SQL;
 - criticidad;
 - umbral.

 ### Importante

 Data Quality evalúa la **calidad y confiabilidad del dato**.

 Una falla DQ:

 - no representa fraude;
 - no constituye una señal Trust & Safety;
 - no incrementa directamente el Risk Score.

In [0]:
# COMMAND ----------

spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS {DQ_RULE_TABLE}
    (
        regla_id       STRING,
        capa           STRING,
        tabla_objetivo STRING,
        columna        STRING,
        dimension      STRING,
        tipo_regla     STRING,
        descripcion    STRING,
        expresion_sql  STRING,
        criticidad     STRING,
        umbral_pct     DOUBLE
    )
    USING DELTA
    """
)

print(
    f"✔ Tabla disponible: {DQ_RULE_TABLE}"
)


# ============================================================
# 4. COMPATIBILIDAD DEL CONTRATO dq_reglas
# ============================================================

expected_columns = {
    "regla_id": "STRING",
    "capa": "STRING",
    "tabla_objetivo": "STRING",
    "columna": "STRING",
    "dimension": "STRING",
    "tipo_regla": "STRING",
    "descripcion": "STRING",
    "expresion_sql": "STRING",
    "criticidad": "STRING",
    "umbral_pct": "DOUBLE"
}

current_columns = {
    field.name
    for field in spark.table(
        DQ_RULE_TABLE
    ).schema.fields
}

missing_columns = [
    f"{column_name} {data_type}"
    for column_name, data_type
    in expected_columns.items()
    if column_name not in current_columns
]

if missing_columns:

    spark.sql(
        f"""
        ALTER TABLE {DQ_RULE_TABLE}
        ADD COLUMNS (
            {", ".join(missing_columns)}
        )
        """
    )

    print(
        "✔ Columnas agregadas: "
        + ", ".join(missing_columns)
    )

else:

    print(
        "✔ dq_reglas cumple el contrato esperado."
    )

# COMMAND ----------



 ## 3 · Historial de resultados — `dq_resultados`

 Esta tabla conserva la evidencia histórica de cada ejecución Data Quality.

 ### Grano

 **1 fila = 1 regla × 1 ejecución DQ**

 Registra:

 - ejecución;
 - timestamp;
 - regla;
 - capa;
 - tabla y columna;
 - dimensión;
 - criticidad;
 - filas evaluadas;
 - filas correctas;
 - filas con error;
 - porcentaje de cumplimiento;
 - umbral;
 - resultado.

 ### Interpretación

 Posteriormente el proceso DQ puede interpretar:

 - `PASS`: cumple y no presenta errores;
 - `WARNING`: cumple el umbral pero presenta errores;
 - `FAIL`: no alcanza el umbral.

In [0]:

# COMMAND ----------


spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS {DQ_RESULT_TABLE}
    (
        ejecucion_id       STRING,
        ejecucion_ts       TIMESTAMP,
        regla_id           STRING,
        capa               STRING,
        tabla_objetivo     STRING,
        columna            STRING,
        dimension          STRING,
        tipo_regla         STRING,
        criticidad         STRING,
        filas_totales      BIGINT,
        filas_ok           BIGINT,
        filas_error        BIGINT,
        pct_cumplimiento   DOUBLE,
        umbral_pct         DOUBLE,
        paso               BOOLEAN
    )
    USING DELTA
    """
)

print(
    f"✔ Tabla disponible: {DQ_RESULT_TABLE}"
)

# COMMAND ----------

 # Trust & Safety

 ## 4 · Catálogo maestro — `evaluation_rule`

 Contiene la identidad funcional de las reglas heurísticas
 Trust & Safety.

 ### Grano

 **1 fila = 1 regla T&S**

 Una regla identifica patrones potencialmente sospechosos.

 **No confirma fraude.**

 El campo `owner` representa el responsable funcional de la regla.

 No representa el `OWNER` técnico de Unity Catalog.


In [0]:

# COMMAND ----------

# ============================================================
# 6. CREAR evaluation_rule
# ============================================================

spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS {RULE_TABLE}
    (
        rule_id              STRING,
        rule_name            STRING,
        rule_type            STRING,
        business_definition  STRING,
        owner                STRING,
        active_flag          BOOLEAN,
        created_at           TIMESTAMP
    )
    USING DELTA
    """
)

print(
    f"✔ Tabla disponible: {RULE_TABLE}"
)

# COMMAND ----------

 ## 5 · Versionado de reglas — `evaluation_rule_version`

 La identidad de una regla se separa de su configuración.

 ```text
 evaluation_rule
       1
       │
       N
 evaluation_rule_version
 ```

 ### Grano

 **1 fila = 1 versión de una regla T&S**

 Una versión conserva:

 - parámetros;
 - threshold;
 - peso;
 - fecha de vigencia;
 - estado.

 Esto permite evolucionar las reglas sin perder trazabilidad histórica.

In [0]:
# COMMAND ----------


spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS {RULE_VERSION_TABLE}
    (
        rule_version_id  STRING,
        rule_id          STRING,
        version          STRING,
        description      STRING,
        parameters_json  STRING,
        threshold        DOUBLE,
        weight           DOUBLE,
        valid_from       TIMESTAMP,
        status           STRING
    )
    USING DELTA
    """
)

print(
    f"✔ Tabla disponible: {RULE_VERSION_TABLE}"
)

# COMMAND ----------

## 6 · Trazabilidad de ejecución — `evaluation_run`

 Representa una ejecución lógica Trust & Safety.

 ### Grano

 **1 fila = 1 ejecución T&S**

 ```text
 Dataset Silver
      ↓
 Evaluation Run
      ↓
 Review × Rule Version
      ↓
 Risk Consolidation
      ↓
 Gold
 ```

 El `evaluation_run_id` mantiene trazabilidad entre evaluación,
 riesgo y publicación.


In [0]:
# COMMAND ----------

spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS {EVALUATION_RUN_TABLE}
    (
        evaluation_run_id  STRING,
        pipeline_name      STRING,
        pipeline_version   STRING,
        dataset_version    STRING,
        started_at         TIMESTAMP,
        finished_at        TIMESTAMP,
        execution_status   STRING
    )
    USING DELTA
    """
)

print(
    f"✔ Tabla disponible: {EVALUATION_RUN_TABLE}"
)

# COMMAND ----------


 # Metadata Unity Catalog

 ## 7 · Comentarios de tablas

 Los objetos deben ser comprensibles directamente desde Unity Catalog.

 Los comentarios permiten identificar:

 - propósito;
 - significado;
 - responsabilidad;
 - diferencia entre DQ y Trust & Safety.

In [0]:
# COMMAND ----------


table_comments = {

    DQ_RULE_TABLE:
        "Catálogo gobernado de reglas Data Quality. Cada registro representa una regla con dimensión, condición técnica, criticidad y umbral. DQ evalúa calidad del dato y no representa fraude ni una señal Trust & Safety.",

    DQ_RESULT_TABLE:
        "Historial de resultados Data Quality. Cada registro representa el resultado de una regla para una ejecución específica.",

    RULE_TABLE:
        "Catálogo maestro de reglas heurísticas Trust & Safety. Las reglas permiten identificar patrones potencialmente sospechosos y no confirman fraude.",

    RULE_VERSION_TABLE:
        "Versionado gobernado de reglas Trust & Safety con parámetros, threshold, peso, vigencia y estado.",

    EVALUATION_RUN_TABLE:
        "Registro de ejecuciones lógicas Trust & Safety utilizado para mantener trazabilidad entre dataset, evaluación, riesgo y productos Gold."
}


for table_name, comment in table_comments.items():

    safe_comment = (
        comment.replace(
            "'",
            "''"
        )
    )

    spark.sql(
        f"""
        COMMENT ON TABLE {table_name}
        IS '{safe_comment}'
        """
    )

    print(
        f"✔ COMMENT aplicado: {table_name}"
    )

# COMMAND ----------

 ## 8 · Comentarios de columnas

 Los comentarios de columnas forman parte del catálogo técnico.

 Permiten comprender el contrato sin necesidad de revisar el código
 de los notebooks consumidores.

In [0]:
# COMMAND ----------


dq_rule_comments = {

    "regla_id":
        "Identificador único de la regla Data Quality.",

    "capa":
        "Capa evaluada por la regla, por ejemplo bronze o silver.",

    "tabla_objetivo":
        "Nombre completamente calificado de la tabla evaluada.",

    "columna":
        "Columna principal evaluada. Puede ser nula cuando la regla aplica a toda la tabla.",

    "dimension":
        "Dimensión Data Quality: completitud, unicidad, validez, consistencia, integridad o actualidad.",

    "tipo_regla":
        "Tipo técnico de evaluación, por ejemplo CONDITION o UNIQUENESS.",

    "descripcion":
        "Descripción funcional de la condición de calidad.",

    "expresion_sql":
        "Expresión SQL utilizada por la regla cuando corresponde.",

    "criticidad":
        "Criticidad de la regla utilizada por el Quality Gate.",

    "umbral_pct":
        "Porcentaje mínimo de cumplimiento requerido."
}


for column_name, comment in dq_rule_comments.items():

    safe_comment = comment.replace(
        "'",
        "''"
    )

    spark.sql(
        f"""
        ALTER TABLE {DQ_RULE_TABLE}
        ALTER COLUMN {column_name}
        COMMENT '{safe_comment}'
        """
    )


print(
    "✔ Column comments: dq_reglas"
)

# COMMAND ----------

In [0]:
# ============================================================
# 11. COLUMN COMMENTS - dq_resultados
# ============================================================

dq_result_comments = {

    "ejecucion_id":
        "Identificador único de la ejecución Data Quality.",

    "ejecucion_ts":
        "Timestamp UTC de la ejecución.",

    "regla_id":
        "Identificador de la regla ejecutada.",

    "capa":
        "Capa del Lakehouse evaluada.",

    "tabla_objetivo":
        "Tabla evaluada.",

    "columna":
        "Columna evaluada cuando corresponde.",

    "dimension":
        "Dimensión de calidad evaluada.",

    "tipo_regla":
        "Tipo técnico de regla ejecutada.",

    "criticidad":
        "Criticidad de la regla.",

    "filas_totales":
        "Cantidad total de filas evaluadas.",

    "filas_ok":
        "Cantidad de filas que cumplen la regla.",

    "filas_error":
        "Cantidad de filas que no cumplen la regla.",

    "pct_cumplimiento":
        "Porcentaje obtenido de cumplimiento.",

    "umbral_pct":
        "Porcentaje mínimo requerido.",

    "paso":
        "Indica si el porcentaje alcanzó el umbral definido."
}


for column_name, comment in dq_result_comments.items():

    safe_comment = comment.replace(
        "'",
        "''"
    )

    spark.sql(
        f"""
        ALTER TABLE {DQ_RESULT_TABLE}
        ALTER COLUMN {column_name}
        COMMENT '{safe_comment}'
        """
    )


print(
    "✔ Column comments: dq_resultados"
)

# COMMAND ----------

In [0]:
# ============================================================
# 12. COLUMN COMMENTS - evaluation_rule
# ============================================================

rule_comments = {

    "rule_id":
        "Identificador único de la regla Trust & Safety.",

    "rule_name":
        "Nombre funcional de la regla.",

    "rule_type":
        "Tipo de patrón analizado por la regla.",

    "business_definition":
        "Definición funcional del patrón evaluado.",

    "owner":
        "Responsable funcional de la regla. No representa el Owner técnico de Unity Catalog.",

    "active_flag":
        "Indica si la regla está habilitada para evaluación productiva.",

    "created_at":
        "Timestamp de creación inicial de la regla."
}


for column_name, comment in rule_comments.items():

    safe_comment = comment.replace(
        "'",
        "''"
    )

    spark.sql(
        f"""
        ALTER TABLE {RULE_TABLE}
        ALTER COLUMN {column_name}
        COMMENT '{safe_comment}'
        """
    )


print(
    "✔ Column comments: evaluation_rule"
)

# COMMAND ---------

In [0]:

# ============================================================
# 13. COLUMN COMMENTS - evaluation_rule_version
# ============================================================

version_comments = {

    "rule_version_id":
        "Identificador único de la versión de una regla.",

    "rule_id":
        "Identificador de la regla padre.",

    "version":
        "Versión funcional de la regla.",

    "description":
        "Descripción de la versión.",

    "parameters_json":
        "Parámetros técnicos utilizados por la versión.",

    "threshold":
        "Umbral principal utilizado por la regla.",

    "weight":
        "Peso utilizado cuando la regla participa en el score heurístico.",

    "valid_from":
        "Timestamp desde el cual se registró la versión.",

    "status":
        "Estado de la versión, por ejemplo ACTIVE o DRAFT."
}


for column_name, comment in version_comments.items():

    safe_comment = comment.replace(
        "'",
        "''"
    )

    spark.sql(
        f"""
        ALTER TABLE {RULE_VERSION_TABLE}
        ALTER COLUMN {column_name}
        COMMENT '{safe_comment}'
        """
    )


print(
    "✔ Column comments: evaluation_rule_version"
)

# COMMAND ----------

In [0]:
# ============================================================
# 14. COLUMN COMMENTS - evaluation_run
# ============================================================

run_comments = {

    "evaluation_run_id":
        "Identificador único de la ejecución Trust & Safety.",

    "pipeline_name":
        "Nombre del pipeline que ejecutó la evaluación.",

    "pipeline_version":
        "Versión de la lógica de evaluación.",

    "dataset_version":
        "Identificador de la versión Silver utilizada como entrada.",

    "started_at":
        "Timestamp UTC de inicio.",

    "finished_at":
        "Timestamp UTC de finalización.",

    "execution_status":
        "Estado de ejecución, por ejemplo RUNNING, SUCCESS o FAILED."
}


for column_name, comment in run_comments.items():

    safe_comment = comment.replace(
        "'",
        "''"
    )

    spark.sql(
        f"""
        ALTER TABLE {EVALUATION_RUN_TABLE}
        ALTER COLUMN {column_name}
        COMMENT '{safe_comment}'
        """
    )


print(
    "✔ Column comments: evaluation_run"
)

# COMMAND ----------

 ## 9 · Tags de Unity Catalog

 Se utiliza una taxonomía consistente:

 - `data_domain`
 - `data_layer`
 - `data_product`
 - `classification`
 - `criticality`

 ### Clasificación

 `dq_reglas`, `dq_resultados` y `evaluation_run`:

 `internal`

 `evaluation_rule` y `evaluation_rule_version`:

 `confidential`

 porque exponen detalles sobre la lógica heurística,
 parámetros, thresholds y pesos.

 Si el entorno no soporta tags, el notebook muestra una advertencia
 pero no invalida el contrato físico.

In [0]:
# COMMAND ----------

# ============================================================
# 15. TAGS UNITY CATALOG
# ============================================================

table_tags = {

    DQ_RULE_TABLE: {
        "data_domain": "data_quality",
        "data_layer": "governance",
        "data_product": "dq_rule_catalog",
        "classification": "internal",
        "criticality": "high"
    },

    DQ_RESULT_TABLE: {
        "data_domain": "data_quality",
        "data_layer": "governance",
        "data_product": "dq_execution_history",
        "classification": "internal",
        "criticality": "high"
    },

    RULE_TABLE: {
        "data_domain": "trust_safety",
        "data_layer": "governance",
        "data_product": "evaluation_rule_catalog",
        "classification": "confidential",
        "criticality": "high"
    },

    RULE_VERSION_TABLE: {
        "data_domain": "trust_safety",
        "data_layer": "governance",
        "data_product": "evaluation_rule_version_catalog",
        "classification": "confidential",
        "criticality": "high"
    },

    EVALUATION_RUN_TABLE: {
        "data_domain": "trust_safety",
        "data_layer": "governance",
        "data_product": "evaluation_run",
        "classification": "internal",
        "criticality": "high"
    }
}


for table_name, tags in table_tags.items():

    tag_sql = ", ".join(
        [
            f"'{key}' = '{value}'"
            for key, value
            in tags.items()
        ]
    )

    try:

        spark.sql(
            f"""
            ALTER TABLE {table_name}
            SET TAGS (
                {tag_sql}
            )
            """
        )

        print(
            f"✔ TAGS aplicados: {table_name}"
        )

    except Exception as exc:

        print(
            f"⚠ No fue posible aplicar TAGS: {table_name}"
        )

        print(
            f"Detalle: {exc}"
        )

# COMMAND ----------

 ## 10 · Owner y seguridad

 En esta etapa no se modifica el Owner técnico.

 Debemos distinguir:

 | Concepto | Responsabilidad |
 |---|---|
 | Data Owner | Responsable funcional |
 | Data Steward | Calidad y definición |
 | Unity Catalog Owner | Control administrativo técnico |

 Los siguientes controles se implementarán posteriormente:

 - `OWNER TO`;
 - `GRANT`;
 - `REVOKE`;
 - secure views;
 - pseudonimización;
 - masking si el entorno lo soporta.

 No se declarará un control como implementado sin evidencia técnica.

# COMMAND ----------


 # Validación del modelo

 Las cinco estructuras esperadas son:

 ```text
 workspace.yelp_gov
 │
 ├── dq_reglas
 ├── dq_resultados
 ├── evaluation_rule
 ├── evaluation_rule_version
 └── evaluation_run
 ```

 En esta etapa las tablas pueden estar vacías.

In [0]:

# COMMAND ----------

# ============================================================
# 16. VALIDAR OBJETOS
# ============================================================

required_tables = [
    DQ_RULE_TABLE,
    DQ_RESULT_TABLE,
    RULE_TABLE,
    RULE_VERSION_TABLE,
    EVALUATION_RUN_TABLE
]


missing_tables = [
    table_name
    for table_name in required_tables
    if not spark.catalog.tableExists(
        table_name
    )
]


if missing_tables:

    print(
        "❌ OBJETOS FALTANTES:"
    )

    for table_name in missing_tables:

        print(
            f" - {table_name}"
        )

    raise Exception(
        "El modelo físico de gobierno quedó incompleto."
    )


print(
    "✔ Todos los objetos de gobierno existen."
)

# COMMAND ----------

In [0]:

# ============================================================
# 17. INVENTARIO UNITY CATALOG
# ============================================================

display(
    spark.sql(
        f"""
        SELECT
            table_schema,
            table_name,
            table_type,
            comment

        FROM
            {CATALOG}.information_schema.tables

        WHERE
            table_schema = '{GOV_SCHEMA}'

        AND table_name IN
        (
            'dq_reglas',
            'dq_resultados',
            'evaluation_rule',
            'evaluation_rule_version',
            'evaluation_run'
        )

        ORDER BY
            table_name
        """
    )
)

# COMMAND ----------


In [0]:
# ============================================================
# 18. VALIDAR TAGS
# ============================================================

try:

    display(
        spark.sql(
            f"""
            SELECT
                schema_name,
                table_name,
                tag_name,
                tag_value

            FROM
                {CATALOG}.information_schema.table_tags

            WHERE
                schema_name = '{GOV_SCHEMA}'

            AND table_name IN
            (
                'dq_reglas',
                'dq_resultados',
                'evaluation_rule',
                'evaluation_rule_version',
                'evaluation_run'
            )

            ORDER BY
                table_name,
                tag_name
            """
        )
    )

except Exception as exc:

    print(
        "⚠ No fue posible consultar information_schema.table_tags."
    )

    print(
        f"Detalle: {exc}"
    )

# COMMAND ----------

 ## Definition of Done

 `02_configurar_modelo_gobierno` se considera completado cuando:

 - [ ] `workspace.yelp_gov` existe;
 - [ ] `dq_reglas` existe;
 - [ ] `dq_resultados` existe;
 - [ ] `evaluation_rule` existe;
 - [ ] `evaluation_rule_version` existe;
 - [ ] `evaluation_run` existe;
 - [ ] las tablas poseen comentarios funcionales;
 - [ ] las columnas poseen comentarios técnicos;
 - [ ] los tags se aplicaron cuando están disponibles;
 - [ ] no se publicaron reglas DQ;
 - [ ] no se publicaron reglas T&S;
 - [ ] no se generaron resultados operativos;
 - [ ] no se modificaron Owners ni permisos.

 ---

 ## Siguiente paso

 ```text
 03_validar_fuentes_raw
         ↓
 E5/01_publicar_catalogo_reglas_dq_silver
         ↓
 E6/02_publicar_catalogo_reglas_ts
 ```

In [0]:
# COMMAND ----------

# ============================================================
# 19. RESUMEN FINAL
# ============================================================

print()
print("=" * 72)
print("MODELO DE GOBIERNO CONFIGURADO")
print("=" * 72)

print()
print("DATA QUALITY")
print(f"✔ {DQ_RULE_TABLE}")
print(f"✔ {DQ_RESULT_TABLE}")

print()
print("TRUST & SAFETY")
print(f"✔ {RULE_TABLE}")
print(f"✔ {RULE_VERSION_TABLE}")
print(f"✔ {EVALUATION_RUN_TABLE}")

print()
print("RESULTADO")
print("✔ Estructuras físicas disponibles")
print("✔ Metadata descriptiva configurada")
print("✔ Validaciones ejecutadas")
print("✔ Sin publicación de reglas")
print("✔ Sin ejecución DQ/T&S")

print()
print("SIGUIENTE PASO")
print("→ 03_validar_fuentes_raw")

print("=" * 72)